# 02 - Feature Engineering and Proxy Label Creation

This notebook creates the features that make the project look like a real risk-intelligence system.

It also creates:
- a **binary risk label** for classification
- a **continuous exposure target** for regression
- **risk-source labels** for multi-label prediction

The labels are proxy targets because public CMS data does not directly contain a `surprise_bill` flag.

In [ ]:

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)

df = pd.read_parquet("data/base_cleaned.parquet")
print(df.shape)
df.head()

In [ ]:

eps = 1e-6

# Core financial features
df["charge_ratio_inpatient"] = df["average_covered_charges"] / (df["average_total_payments"] + eps)
df["payment_gap_inpatient"] = df["average_covered_charges"] - df["average_total_payments"]

df["charge_ratio_outpatient"] = df["Billed amount"] / (df["Medicare payment"] + eps)
df["payment_gap_outpatient"] = df["Billed amount"] - df["Medicare payment"]

# Blended features to handle mixed service records
df["blended_charge_ratio"] = np.where(
    df["charge_ratio_outpatient"].notna(),
    df["charge_ratio_outpatient"],
    df["charge_ratio_inpatient"]
)
df["blended_payment_gap"] = np.where(
    df["payment_gap_outpatient"].notna(),
    df["payment_gap_outpatient"],
    df["payment_gap_inpatient"]
)

# Log-scaled magnitude features
df["log_avg_covered"] = np.log1p(df["average_covered_charges"])
df["log_avg_payments"] = np.log1p(df["average_total_payments"])
df["log_billed_amount"] = np.log1p(df["Billed amount"])
df["log_medicare_payment"] = np.log1p(df["Medicare payment"])
df["log_blended_gap"] = np.log1p(np.clip(df["blended_payment_gap"], a_min=0, a_max=None))

In [ ]:

# DRG and service heuristics
def contains_any(text: str, keywords: list[str]) -> int:
    text = str(text).lower()
    return int(any(k in text for k in keywords))

df["is_er"] = df["service_type"].apply(lambda x: contains_any(x, ["er", "emergency"]))
df["is_imaging"] = df["service_type"].apply(lambda x: contains_any(x, ["imaging", "radiology", "scan", "mri", "ct", "x-ray", "xray"]))
df["is_outpatient"] = df["service_type"].apply(lambda x: contains_any(x, ["outpatient", "clinic", "ambulatory", "same day"]))
df["is_surgery"] = (
    df["service_type"].apply(lambda x: contains_any(x, ["surgery", "operative", "procedure"]))
    | df["drg_text"].apply(lambda x: contains_any(x, ["surgery", "replacement", "fusion", "bypass", "repair"]))
).astype(int)

df["anesthesia_flag"] = (
    df["service_type"].apply(lambda x: contains_any(x, ["anesthesia"]))
    | df["drg_text"].apply(lambda x: contains_any(x, ["anesthesia"]))
    | df["is_surgery"]
).astype(int)

df["radiology_flag"] = (
    df["is_imaging"]
    | df["drg_text"].apply(lambda x: contains_any(x, ["imaging", "radiology", "ct", "mri"]))
).astype(int)

df["pathology_flag"] = (
    df["service_type"].apply(lambda x: contains_any(x, ["lab", "pathology", "specimen"]))
    | df["drg_text"].apply(lambda x: contains_any(x, ["biopsy", "tumor", "neoplasm", "pathology"]))
    | df["is_surgery"]
).astype(int)

df["high_complexity_drg"] = df["drg_text"].apply(
    lambda x: contains_any(
        x,
        ["cardiac", "bypass", "transplant", "major", "joint replacement", "spinal", "fusion", "craniotomy", "ventilator"]
    )
)

df[[
    "is_er", "is_imaging", "is_outpatient", "is_surgery",
    "anesthesia_flag", "radiology_flag", "pathology_flag", "high_complexity_drg"
]].mean().sort_values(ascending=False)

In [ ]:

# Provider and state aggregates
provider_agg = (
    df.groupby("provider_id")
      .agg(
          provider_avg_gap=("blended_payment_gap", "mean"),
          provider_gap_std=("blended_payment_gap", "std"),
          provider_avg_ratio=("blended_charge_ratio", "mean"),
          provider_record_count=("provider_id", "size")
      )
      .reset_index()
)

state_agg = (
    df.groupby("provider_state")
      .agg(
          state_avg_gap=("blended_payment_gap", "mean"),
          state_avg_ratio=("blended_charge_ratio", "mean"),
          state_median_payment=("average_total_payments", "median"),
          state_record_count=("provider_state", "size")
      )
      .reset_index()
)

df = df.merge(provider_agg, on="provider_id", how="left")
df = df.merge(state_agg, on="provider_state", how="left")
df.head()

In [ ]:

# Normalized indices for presentation
def minmax(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    denom = (s.max() - s.min())
    if denom == 0:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - s.min()) / denom

df["provider_risk_index"] = 0.6 * minmax(df["provider_avg_gap"]) + 0.4 * minmax(df["provider_avg_ratio"])
df["state_risk_index"] = 0.6 * minmax(df["state_avg_gap"]) + 0.4 * minmax(df["state_avg_ratio"])
df["service_risk_weight"] = (
    0.35 * df["is_er"] +
    0.25 * df["is_surgery"] +
    0.15 * df["is_imaging"] +
    0.10 * df["pathology_flag"] +
    0.15 * df["high_complexity_drg"]
)
df["service_risk_weight"] = df["service_risk_weight"].clip(0, 1)

In [ ]:

# Proxy targets
ratio_thr = df["blended_charge_ratio"].quantile(0.70)
gap_thr = df["blended_payment_gap"].quantile(0.70)
provider_thr = df["provider_risk_index"].quantile(0.70)
state_thr = df["state_risk_index"].quantile(0.70)

high_ratio = (df["blended_charge_ratio"] >= ratio_thr).astype(int)
high_gap = (df["blended_payment_gap"] >= gap_thr).astype(int)
high_provider = (df["provider_risk_index"] >= provider_thr).astype(int)
high_state = (df["state_risk_index"] >= state_thr).astype(int)

risk_points = (
    2 * high_ratio +
    2 * high_gap +
    1 * df["is_er"] +
    1 * df["is_surgery"] +
    1 * df["radiology_flag"] +
    1 * df["pathology_flag"] +
    1 * high_provider +
    1 * high_state +
    1 * df["high_complexity_drg"]
)

df["proxy_surprise_bill_label"] = (risk_points >= 5).astype(int)

# Continuous target for exposure regression
df["proxy_exposure_amount"] = (
    0.45 * np.clip(df["blended_payment_gap"], a_min=0, a_max=None) +
    0.30 * np.clip(df["provider_avg_gap"], a_min=0, a_max=None) +
    0.15 * np.clip(df["state_avg_gap"], a_min=0, a_max=None) +
    500 * df["is_er"] +
    700 * df["is_surgery"] +
    300 * df["high_complexity_drg"]
)

df["proxy_exposure_amount"] = df["proxy_exposure_amount"].clip(lower=0)

df[["proxy_surprise_bill_label", "proxy_exposure_amount"]].describe()

In [ ]:

# Simple presentation plots
plt.figure(figsize=(8, 5))
df["proxy_surprise_bill_label"].value_counts().sort_index().plot(kind="bar")
plt.title("Proxy Risk Label Distribution")
plt.xlabel("Label (0 = lower risk, 1 = higher risk)")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()

In [ ]:

top_states = (
    df.groupby("provider_state")["proxy_surprise_bill_label"]
      .mean()
      .sort_values(ascending=False)
      .head(15)
)
plt.figure(figsize=(10, 5))
top_states.plot(kind="bar")
plt.title("Top States by Average Proxy Risk")
plt.xlabel("State")
plt.ylabel("Average Risk")
plt.tight_layout()
plt.show()

In [ ]:

feature_cols = [
    "provider_id", "provider_name", "provider_state", "drg_definition", "service_type", "drg_text",
    "average_covered_charges", "average_total_payments", "Billed amount", "Medicare payment",
    "charge_ratio_inpatient", "payment_gap_inpatient", "charge_ratio_outpatient", "payment_gap_outpatient",
    "blended_charge_ratio", "blended_payment_gap", "log_avg_covered", "log_avg_payments",
    "log_billed_amount", "log_medicare_payment", "log_blended_gap",
    "is_er", "is_imaging", "is_outpatient", "is_surgery",
    "anesthesia_flag", "radiology_flag", "pathology_flag", "high_complexity_drg",
    "provider_avg_gap", "provider_gap_std", "provider_avg_ratio", "provider_record_count",
    "state_avg_gap", "state_avg_ratio", "state_median_payment", "state_record_count",
    "provider_risk_index", "state_risk_index", "service_risk_weight",
    "proxy_surprise_bill_label", "proxy_exposure_amount"
]

model_df = df[feature_cols].copy()
model_df.to_parquet("data/model_features.parquet", index=False)
print("Saved to data/model_features.parquet")